# Zadanie 1, część I: anotacja korpusu PolEmo 2.0 (procedura 2+1)

1. wczytanie korpusu i przygotowanie danych do anotacji,
2. wytyczne anotacji i eksport arkuszy dla anotatorów (Google Sheets),
3. zgodność po iteracji 1 (Kappa Cohena, Kappa Fleissa),
4. nowe wytyczne i zgodność po iteracji 2, porównanie iteracji,
5. anotacja reszty zbioru,
6. statystyki opisowe zebranego zbioru.

**Procedura 2+1:** dwie osoby (A, B) anotują te same zdania niezależnie, trzecia osoba (C) rozstrzyga rozbieżności.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from IPython.display import display
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font
from openpyxl.worksheet.datavalidation import DataValidation
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

SEED = 42
N_PER_ITER = 50                 # zdań na iterację
N_ROZNE = 150                   # krok 5: zdań na osobę przy anotacji reszty
ANNOTATORS = ["A", "B"]         # dwie osoby anotują niezależnie, C rozstrzyga konflikty
MIN_WORDS, MAX_WORDS = 4, 40    # filtr długości zdań

OUT = Path("anotacja")
OUT.mkdir(exist_ok=True)

SENT_LABELS = ["POZ", "NEG", "NEU", "AMB"]
GOLD_MAP = {"z_plus_m": "POZ", "z_minus_m": "NEG", "z_zero": "NEU", "z_amb": "AMB"}
MOD_TYPES = ["WZM", "OSL", "ODW"]

pd.set_option("display.max_colwidth", 120)

## 1. Wczytanie korpusu

Używamy zdaniowej wersji PolEmo 2.0 (ok. 46 tys. zdań). Każda linia kończy się etykietą, np. `__label__z_minus_m`, którą odcinamy przez `rsplit(" __label__", 1)`. Teksty są już stokenizowane, stąd formy typu "Była m", "mieli śmy".

Etykiety korpusu: `z_plus_m` (pozytywne), `z_minus_m` (negatywne), `z_zero` (neutralne), `z_amb` (ambiwalentne).

In [2]:
ds = load_dataset(
    "text",
    data_files="hf://datasets/clarin-pl/polemo2-official/data/all.sentence.train.txt",
)["train"]


def split_label(ex):
    line = ex["text"].strip()
    if " __label__" in line:
        text, label = line.rsplit(" __label__", 1)
    else:
        text, label = line, None
    return {"text": text.strip(), "gold_raw": label}


df = ds.map(split_label).to_pandas()
df = df[df["gold_raw"].notna() & (df["text"] != "")].reset_index(drop=True)
df["gold"] = df["gold_raw"].map(GOLD_MAP)
df["n_words"] = df["text"].str.split().str.len()

print(f"Wczytano {len(df)} zdań")
display(df.head())
display(df["gold_raw"].value_counts().to_frame("liczba"))
display(df["n_words"].describe().to_frame("liczba słów"))

Wczytano 45974 zdań


,text,gold_raw,gold,n_words
0,"Taksowak przyjezdza na czas , wydaje sie , ze jest , to zupelnie przypadkowa , jednorazowa wpadka .",z_zero,NEU,18
1,Do tej pani trafili śmy z naszą 2 - miesięczna córeczką z podejrzeniem niedojrzałości układu nerwowego .,z_zero,NEU,17
2,"Wyróżnikiem , a także ciekawym elementem całego wystroju są umieszczone w centralnym miejscu regały z książkami , tw...",z_zero,NEU,21
3,"W zależności od pory dnia , w naszych organizmach dochodzi bowiem do bardzo dużych zmian w układzie krążenia : w ciś...",z_zero,NEU,31
4,"Polecam wszystkim , zwłaszcza tym , dla których wizyta u lekarza to duży stres .",z_plus_m,POZ,15


,liczba
gold_raw,
z_minus_m,16911
z_plus_m,12293
z_zero,11255
z_amb,5515


,liczba słów
count,45974.000000
mean,17.869035
std,13.553874
min,1.000000
25%,9.000000
50%,15.000000
75%,23.000000
max,205.000000


### Losowanie próbek

Odrzucamy zdania bardzo krótkie i bardzo długie oraz duplikaty, mieszamy pulę ze stałym ziarnem i dzielimy ją na rozłączne części:

- **iteracja 1**: `N_PER_ITER` zdań, anotują A i B,
- **iteracja 2**: kolejne `N_PER_ITER` nowych zdań, anotują A i B,
- **reszta**: pula do kroku 5 (każda osoba anotuje inne zdania).

Etykiety PolEmo zapisujemy w osobnych plikach `klucz_*.csv`. Anotatorzy ich nie widzą.

In [3]:
pool = (
    df[(df["n_words"] >= MIN_WORDS) & (df["n_words"] <= MAX_WORDS)]
    .drop_duplicates("text")
    .sample(frac=1.0, random_state=SEED)   # losowa kolejność = losowa próbka
    .reset_index(drop=True)
)
pool["id"] = [f"s{i:05d}" for i in range(len(pool))]

parts = {
    "iter1": pool.iloc[:N_PER_ITER],
    "iter2": pool.iloc[N_PER_ITER: 2 * N_PER_ITER],
    "reszta": pool.iloc[2 * N_PER_ITER:],
}

for name, part in parts.items():
    path = OUT / f"klucz_{name}.csv"
    if not path.exists():
        part[["id", "text", "gold_raw", "gold", "n_words"]].to_csv(path, index=False, encoding="utf-8-sig")

print({k: len(v) for k, v in parts.items()})
print("Rozkład etykiet PolEmo w iteracji 1 i 2:")
display(pd.DataFrame({k: parts[k]["gold"].value_counts() for k in ["iter1", "iter2"]}))

{'iter1': 50, 'iter2': 50, 'reszta': 42402}
Rozkład etykiet PolEmo w iteracji 1 i 2:


,iter1,iter2
gold,,
AMB,4,1
NEG,20,20
NEU,9,17
POZ,17,12


## 2. Wytyczne anotacji, wersja 1

**Anotacja całego tekstu, kolumna `sentyment`.** Jedna etykieta na zdanie:

| Etykieta | Kiedy | Przykład |
|---|---|---|
| **POZ** | zdanie wyraża ocenę pozytywną | "Obsługa była bardzo miła." |
| **NEG** | zdanie wyraża ocenę negatywną | "Pokój śmierdział stęchlizną." |
| **NEU** | brak oceny, sam fakt lub opis | "Hotel znajduje się przy dworcu." |
| **AMB** | oceny pozytywne i negatywne o podobnej wadze | "Jedzenie dobre, ale drogie." |

- Oceniamy samo zdanie, bez domyślania się kontekstu całej recenzji.
- Nie zaglądamy do arkusza drugiej osoby i nie konsultujemy decyzji w trakcie anotacji.
- Wątpliwości zapisujemy w kolumnie `uwagi`.

**Anotacja fragmentów, kolumna `modyfikatory`.** Oznaczamy słowa i frazy, które zmieniają siłę lub kierunek oceny:

| Typ | Znaczenie | Przykłady |
|---|---|---|
| **WZM** | wzmocnienie | bardzo, naprawdę, zdecydowanie, ogromnie |
| **OSL** | osłabienie | trochę, raczej, dość, nieco |
| **ODW** | odwrócenie polaryzacji | nie, wcale nie, brak, bez |

- Format: `fraza=TYP; fraza=TYP`, np. `bardzo=WZM; nie=ODW`.
- Frazę kopiujemy dokładnie tak, jak w tekście (także rozbite formy typu "Była m").
- Jeśli fraza występuje w zdaniu kilka razy: `nie@2=ODW` oznacza drugie wystąpienie.
- Oznaczamy sam modyfikator, nie słowo, które on modyfikuje.
- Brak modyfikatorów: komórkę zostawiamy pustą.

### Eksport arkuszy dla anotatorów

Każdy arkusz ma kolumny `id`, `text`, `sentyment` (lista rozwijana), `modyfikatory`, `uwagi`. Każda osoba ma swój plik do anotacji.

In [4]:
def write_sheet(part: pd.DataFrame, path: Path):
    if path.exists():
        print(f"pomijam (już istnieje): {path.name}")
        return
    wb = Workbook()
    ws = wb.active
    ws.title = "anotacja"
    ws.append(["id", "text", "sentyment", "modyfikatory", "uwagi"])
    for c in ws[1]:
        c.font = Font(bold=True)
    for _, r in part.iterrows():
        ws.append([r["id"], r["text"], None, None, None])

    dv = DataValidation(type="list", formula1='"' + ",".join(SENT_LABELS) + '"',
                        allow_blank=False, showErrorMessage=True)
    ws.add_data_validation(dv)
    dv.add(f"C2:C{len(part) + 1}")

    for col, w in zip("ABCDE", [9, 90, 12, 40, 30]):
        ws.column_dimensions[col].width = w
    for row in ws.iter_rows(min_row=2, min_col=2, max_col=2):
        row[0].alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "C2"
    wb.save(path)
    print(f"zapisano: {path.name}")


for it in ["iter1", "iter2"]:
    for a in ANNOTATORS:
        write_sheet(parts[it], OUT / f"{it}_anotator_{a}.xlsx")

# Krok 5: rozłączne paczki z reszty, każda osoba anotuje inne zdania
for i, a in enumerate(ANNOTATORS + ["C"]):
    write_sheet(parts["reszta"].iloc[i * N_ROZNE:(i + 1) * N_ROZNE], OUT / f"reszta_anotator_{a}.xlsx")

pomijam (już istnieje): iter1_anotator_A.xlsx
pomijam (już istnieje): iter1_anotator_B.xlsx
pomijam (już istnieje): iter2_anotator_A.xlsx
pomijam (już istnieje): iter2_anotator_B.xlsx
pomijam (już istnieje): reszta_anotator_A.xlsx
pomijam (już istnieje): reszta_anotator_B.xlsx
pomijam (już istnieje): reszta_anotator_C.xlsx


## 3. Zgodność anotatorów

Liczymy zgodność na dwóch poziomach:

- **cały tekst**: etykiety z kolumny `sentyment`,
- **fragmenty**: frazy z kolumny `modyfikatory` zamieniamy na etykiety tokenów (`O`/`WZM`/`OSL`/`ODW`) i liczymy kappę na tokenach. Kappa na tokenach jest zawyżana przez dominującą klasę `O`, więc dodatkowo podajemy F1 między A i B liczone tylko na oznaczonych tokenach.

**Kappa Fleissa przy dwóch anotatorach** daje prawie to samo co Kappa Cohena: obie poprawiają zgodność obserwowaną o zgodność przypadkową, a różnią się tylko sposobem jej szacowania (Cohen: osobne rozkłady etykiet każdego anotatora, Fleiss: wspólny rozkład).

Rozbieżności trafiają do pliku `iterX_rozbieznosci.xlsx`. Osoba C wpisuje w nim decyzję w kolumnie `decyzja_C`.

In [5]:
def load_ann(path):
    d = pd.read_excel(path, dtype=str).fillna("")
    d["sentyment"] = d["sentyment"].str.strip().str.upper()
    return d.set_index("id")


def fleiss(columns):
    table, _ = aggregate_raters(np.column_stack(columns))
    return fleiss_kappa(table, method="fleiss")


def parse_mods(cell):
    out = []
    for item in str(cell).split(";"):
        item = item.strip()
        if not item:
            continue
        if "=" not in item:
            out.append((item, None, 1))
            continue
        phrase, typ = item.rsplit("=", 1)
        occ = 1
        if "@" in phrase:
            phrase, occ_s = phrase.rsplit("@", 1)
            occ = int(occ_s) if occ_s.strip().isdigit() else 1
        out.append((phrase.strip(), typ.strip().upper(), occ))
    return out


def token_labels(text, cell, warn_prefix=""):
    toks = text.split()
    labels = ["O"] * len(toks)
    for phrase, typ, occ in parse_mods(cell):
        ptoks = [p.lower() for p in phrase.split()]
        if typ not in MOD_TYPES:
            print(f"{warn_prefix} zły typ: '{phrase}={typ}'")
            continue
        starts = [i for i in range(len(toks) - len(ptoks) + 1)
                  if [t.lower() for t in toks[i:i + len(ptoks)]] == ptoks]
        if len(starts) < occ:
            print(f"{warn_prefix} nie znaleziono frazy '{phrase}' (wystąpienie {occ})")
            continue
        s = starts[occ - 1]
        for j in range(s, s + len(ptoks)):
            labels[j] = typ
    return labels


def agreement(it):
    files = {k: OUT / f"{it}_{k}_gotowe.xlsx" for k in ["A", "B", "C"]}
    if not (files["A"].exists() and files["B"].exists()):
        print(f"Brak plików {files['A'].name} / {files['B'].name}: najpierw zaanotujcie {it}.")
        return None
    ann = {k: load_ann(p) for k, p in files.items() if p.exists()}
    names = list(ann)
    ids = ann["A"].index

    for k, d in ann.items():
        s = d.loc[ids, "sentyment"]
        print(f"{k}: puste etykiety={s.eq('').sum()}, niepoprawne={(~s.isin(SENT_LABELS + [''])).sum()}")

    res = {}
    # --- poziom tekstu ---
    a, b = ann["A"].loc[ids, "sentyment"], ann["B"].loc[ids, "sentyment"]
    res["tekst: zgodność % A-B"] = (a == b).mean()
    res["tekst: Kappa Cohena A-B"] = cohen_kappa_score(a, b)
    res["tekst: Kappa Fleissa A,B"] = fleiss([a, b])
    if "C" in ann:
        c = ann["C"].loc[ids, "sentyment"]
        res["tekst: Kappa Cohena A-C"] = cohen_kappa_score(a, c)
        res["tekst: Kappa Cohena B-C"] = cohen_kappa_score(b, c)
        res["tekst: Kappa Fleissa A,B,C"] = fleiss([a, b, c])

    key = pd.read_csv(OUT / f"klucz_{it}.csv").set_index("id").loc[ids, "gold"]
    for k in names:
        res[f"tekst: Kappa Cohena {k}-PolEmo"] = cohen_kappa_score(ann[k].loc[ids, "sentyment"], key)

    print(f"\nMacierz pomyłek {it} (wiersze A, kolumny B):")
    display(pd.DataFrame(confusion_matrix(a, b, labels=SENT_LABELS),
                         index=[f"A:{l}" for l in SENT_LABELS], columns=[f"B:{l}" for l in SENT_LABELS]))

    # --- poziom fragmentów ---
    tok = {k: [] for k in names}
    spans = {k: set() for k in names}
    for sid in ids:
        text = ann["A"].loc[sid, "text"]
        for k in names:
            labs = token_labels(text, ann[k].loc[sid, "modyfikatory"], warn_prefix=f"[{k} {sid}]")
            tok[k].extend(labs)
            spans[k] |= {(sid, i, l) for i, l in enumerate(labs) if l != "O"}

    res["fragmenty: Kappa Cohena A-B (tokeny)"] = cohen_kappa_score(tok["A"], tok["B"])
    res["fragmenty: Kappa Fleissa A,B (tokeny)"] = fleiss([tok["A"], tok["B"]])
    if "C" in ann:
        res["fragmenty: Kappa Fleissa A,B,C (tokeny)"] = fleiss([tok[k] for k in names])
    inter = len(spans["A"] & spans["B"])
    p, r = inter / max(len(spans["B"]), 1), inter / max(len(spans["A"]), 1)
    res["fragmenty: F1 A-B (bez O)"] = 2 * p * r / max(p + r, 1e-9)
    for k in names:
        res[f"fragmenty: oznaczone tokeny {k}"] = len(spans[k])

    # --- rozbieżności dla osoby C ---
    dis_path = OUT / f"{it}_rozbieznosci.xlsx"
    dis = pd.DataFrame({"text": ann["A"].loc[ids, "text"], "A": a, "B": b,
                        "uwagi_A": ann["A"].loc[ids, "uwagi"], "uwagi_B": ann["B"].loc[ids, "uwagi"]})
    dis = dis[dis["A"] != dis["B"]]
    if dis_path.exists():
        print(f"\n{dis_path.name} już istnieje, nie nadpisuję (mogą tam być decyzje C).")
    else:
        dis.assign(decyzja_C="").to_excel(dis_path)
        print(f"\nZapisano {len(dis)} rozbieżności do {dis_path.name}")
    display(dis)

    out = pd.Series(res, name=it).round(3)
    display(out.to_frame())
    return out

def decyzje_C(it):
    dis_path = OUT / f"{it}_rozbieznosci.xlsx"
    if not dis_path.exists():
        print(f"Brak pliku {dis_path.name}: najpierw uruchomcie agreement('{it}').")
        return None
    dis = pd.read_excel(dis_path, dtype=str).fillna("").set_index("id")
    for col in ["A", "B", "decyzja_C"]:
        dis[col] = dis[col].str.strip().str.upper()
    dec = dis["decyzja_C"]
    dis["C zgodna z"] = np.select(
        [dec == "", dec == dis["A"], dec == dis["B"]],
        ["brak decyzji", "A", "B"],
        default="żadnym",
    )

    print(f"Decyzje osoby C w {it} ({len(dis)} rozbieżności):")
    display(dis[["text", "A", "B", "decyzja_C", "C zgodna z", "uwagi_A", "uwagi_B"]])
    display(dis["C zgodna z"].value_counts().to_frame("liczba"))
    display(pd.crosstab(dis["A"] + " / " + dis["B"], dis["decyzja_C"],
                        rownames=["A / B"], colnames=["decyzja C"]))
    return dis


In [6]:
wyniki_iter1 = agreement("iter1")

A: puste etykiety=0, niepoprawne=0
B: puste etykiety=1, niepoprawne=0

Macierz pomyłek iter1 (wiersze A, kolumny B):


,B:POZ,B:NEG,B:NEU,B:AMB
A:POZ,15,0,0,0
A:NEG,0,20,0,0
A:NEU,0,3,6,2
A:AMB,0,2,0,1


[A s00024] nie znaleziono frazy 'brak' (wystąpienie 1)

iter1_rozbieznosci.xlsx już istnieje, nie nadpisuję (mogą tam być decyzje C).


,text,A,B,uwagi_A,uwagi_B
id,,,,,
s00011,Sprzątanie sprowadzało się do wymiany ręczników oraz opróżnienia koszy .,NEU,,,
s00015,w trakcie pracy pojawiają się nieoczekiwane wyjątki choć jeszcze nie stracił em danych .,AMB,NEG,,
s00021,Jeżeli jest sztyca umieszczona pionowo ( a tak wynika ze specyfikacji producenta ) to powinna działać bez zarzutu .,NEU,NEG,,
s00024,Mimo braku pryszniców .,NEU,AMB,,
s00027,"Jeśli zapomni się go ponownie ustawić , to powstaje następujące zjawisko .",NEU,AMB,,
s00039,Internet płatny ( 4 euro za godz . lub 30 za tydzień ) - ale ostatecznie to mają być wakacje .,AMB,NEG,,
s00042,"jest to przecież środek chemiczny usuwający kamień , a więc inwazyjny .",NEU,NEG,,
s00048,"Do ucha trąbka i pogrzebanie haczykiem drucianym , na koniec antybiotyk .",NEU,NEG,,


,iter1
tekst: zgodność % A-B,0.840
tekst: Kappa Cohena A-B,0.764
"tekst: Kappa Fleissa A,B",0.762
tekst: Kappa Cohena A-PolEmo,0.643
tekst: Kappa Cohena B-PolEmo,0.783
fragmenty: Kappa Cohena A-B (tokeny),0.739
"fragmenty: Kappa Fleissa A,B (tokeny)",0.739
fragmenty: F1 A-B (bez O),0.746
fragmenty: oznaczone tokeny A,26.000
fragmenty: oznaczone tokeny B,33.000


In [7]:
decyzje_iter1 = decyzje_C("iter1")

Decyzje osoby C w iter1 (8 rozbieżności):


,text,A,B,decyzja_C,C zgodna z,uwagi_A,uwagi_B
id,,,,,,,
s00011,Sprzątanie sprowadzało się do wymiany ręczników oraz opróżnienia koszy .,NEU,,NEU,A,,
s00015,w trakcie pracy pojawiają się nieoczekiwane wyjątki choć jeszcze nie stracił em danych .,AMB,NEG,AMB,A,,
s00021,Jeżeli jest sztyca umieszczona pionowo ( a tak wynika ze specyfikacji producenta ) to powinna działać bez zarzutu .,NEU,NEG,NEU,A,,
s00024,Mimo braku pryszniców .,NEU,AMB,NEU,A,,
s00027,"Jeśli zapomni się go ponownie ustawić , to powstaje następujące zjawisko .",NEU,AMB,NEU,A,,
s00039,Internet płatny ( 4 euro za godz . lub 30 za tydzień ) - ale ostatecznie to mają być wakacje .,AMB,NEG,AMB,A,,
s00042,"jest to przecież środek chemiczny usuwający kamień , a więc inwazyjny .",NEU,NEG,NEU,A,,
s00048,"Do ucha trąbka i pogrzebanie haczykiem drucianym , na koniec antybiotyk .",NEU,NEG,NEU,A,,


,liczba
C zgodna z,
A,8


decyzja C,AMB,NEU
A / B,,
AMB / NEG,2,0
NEU /,0,1
NEU / AMB,0,2
NEU / NEG,0,3


### Analiza rozbieżności po iteracji 1

W ramach pierwszej iteracji procedury anotacji ($2+1$) dwie niezależne osoby (anotator A oraz anotator B) przeanalizowały tę samą próbkę 50 zdań z korpusu PolEmo 2.0. Uzyskane wyniki wskazują na wysoki poziom zgodności początkowej:
* **Zgodność procentowa (Agreement):** 84.0%
* **Kappa Cohena (poziom tekstu):** 0.764, co wg skali Landisa i Kocha klasyfikuje zgodność jako **znaczącą (substantial)**
* **Kappa Fleissa:** 0.762
* **Zgodność na poziomie modyfikatorów (Kappa Cohena na tokenach):** 0.739 (przy wskaźniku F1 równym 0.746)

W procesie weryfikacji wykryto łącznie 8 rozbieżności (w tym 7 różnic klasyfikacyjnych widocznych w macierzy pomyłek oraz 1 przypadek pustego pola), które zostały wyeksportowane do pliku `iter1_rozbieznosci.xlsx` w celu rozstrzygnięcia przez arbitrów (osobę C).

Odpowiedzi na pytania analityczne:

1. **Które pary klas myliły się najczęściej?**
   Analiza macierzy pomyłek wykazuje, że najwięcej nieporozumień generowała para **NEU (neutralny)** oraz **AMB (ambiwalentny)** lub też **NEG (negatywny)**. Przykładowo, zdar sytuacje, gdzie anotator A ocenił wypowiedź jako obiektywny opis faktyczny (`NEU`), podczas gdy anotator B dopatrzył się w nim mieszanych uczuć (`AMB`) lub ukrytej krytyki (`NEG`). Klasy czysto pozytywne (`POZ`) i negatywne (`NEG`) charakteryzowały się niemal stuprocentową spójnością (brak pomyłek krzyżowych między nimi).

2. **Jakie typy zdań sprawiały największą trudność?**
   Największe trudności analityczne dotyczyły:
   * **Zdań wielowątkowych i zawiłych:** Posiadających wtrącenia techniczne w nawiasach lub zdania złożone, w których obiektywny fakt mieszał się z drobną subiektywną uwagą klienta.
   * **Braków kontekstu:** Krótkie zdania wyrwane z szerszego nurtu recenzji, które bez znajomości całego akapitu mogły być interpretowane dwuznacznie (np. jako neutralna informacja albo zawoalowane zastrzeżenie).

3. **Czy anotatorzy różnie rozumieli modyfikatory?**
   Większość modyfikatorów oznaczono zgodnie z wytycznymi, jednak zauważono rozbieżności w interpretacji słów pełniących funkcję negacji lub osłabienia (np. kwalifikowanie słów takich jak "brak" czy partykuły "nie"). Część wątpliwości wynikała z faktu, że modyfikatory w tekście stokenizowanym bywają nieintuicyjnie podzielone, co utrudniało ich precyzyjne dopasowanie.

**Wnioski dla iteracji 2:**
Wysoka wartość wskaźnika Kappa (>0.75) potwierdza, że ogólne wytyczne są poprawne, jednak aby zminimalizować liczbę rozbieżności w kolejnym etapie, konieczne jest **doprecyzowanie granicy między klasą NEU a AMB** oraz uściślenie zasad oznaczania specyficznych modyfikatorów.

## 4. Wytyczne, wersja 2, i iteracja 2 [1 pkt + 0.5 pkt]

### Zasady ogólne

- **Każdy wiersz musi mieć etykietę.** W razie wątpliwości wybierz najbardziej prawdopodobną i opisz problem w kolumnie `uwagi` (w iteracji 1 jedno zdanie zostało bez etykiety: s00011).
- **Oceniamy wypowiedź autora, a nie opisaną rzecz.** Pytanie pomocnicze: *"Czy autor coś tu chwali albo na coś narzeka?"* Jeśli odpowiedź wymaga dopowiadania, zdanie jest NEU.
- **Oceniamy tylko to, co jest w zdaniu.** Nie domyślamy się kontekstu recenzji. Jeśli zdanie jest urwane, a ocena byłaby w brakującej części, etykieta to NEU (s00024 "Mimo braku pryszniców .", s00027 "…to powstaje następujące zjawisko .").
- **AMB nie oznacza nie wiem.** Niepewność zapisujemy w `uwagi`, a nie przez wybór AMB.

### Etykiety całego tekstu

| Etykieta | Kiedy | Przykład z korpusu |
|---|---|---|
| **POZ** | autor wyraża ocenę pozytywną | "Mysz … wykonana oczywiście bardzo dobrze ." (s00037) |
| **NEG** | autor wyraża ocenę negatywną | "Jedzenie w restauracji przecietne i drogie ." (s00019) |
| **NEU** | fakt, opis, przebieg zdarzeń bez oceny autora | "Jestem ponad rok po operacji korekty wargi i nosa ." (s00010) |
| **AMB** | ocena negatywna i pozytywna (lub łagodząca) połączone kontrastem | "Co prawda zabieg jest drogi , cieszę się , że …" (s00000) |

### Reguły rozstrzygające

**R1. Ocena jawna a domniemana (NEU czy NEG/POZ).**
- **NEG/POZ**, gdy w zdaniu jest słowo oceniające (*słabo, minus, plus, zadowoleni, nieuprzejmy*) albo opis konkretnego doświadczenia autora z jednoznacznym skutkiem: awaria (s00023 "…spalila sie"), czekanie (s00002), niespełnione oczekiwanie (s00035 "spodziewałam się, że lekarz…"), rezygnacja z powrotu (s00020 "następnym razem napewno wybierzemy inny hotel").
- **NEU**, gdy zdanie opisuje cechy, procedurę lub przebieg zdarzeń bez oceny autora, nawet jeśli czytelnik uznałby to za nieprzyjemne: s00048 (opis zabiegu laryngologicznego), s00042 (opis działania środka chemicznego), s00011 (zakres sprzątania).

**R2. Zdania warunkowe i hipotetyczne.**
- **NEU**, gdy zdanie opisuje warunek, instrukcję lub oczekiwanie, a nie rzeczywiste doświadczenie: s00021 "Jeżeli jest sztyca umieszczona pionowo … to powinna działać bez zarzutu".
- **Oceniamy**, gdy warunek opisuje rzeczywiste, regularne zachowanie ocenianej osoby lub rzeczy: s00034 "Jeśli nie zaliczyłeś na I terminie to bierze pod uwagę obecności" -> POZ.

**R3. AMB czy NEG/POZ.**
- **AMB**, gdy ocena negatywna i element pozytywny lub łagodzący są połączone kontrastem (*ale, choć, co prawda, mimo to, z drugiej strony*): s00015 "pojawiają się nieoczekiwane wyjątki choć jeszcze nie straciłem danych", s00039 "Internet płatny … ale ostatecznie to mają być wakacje".
- **Nie AMB**, gdy w zdaniu jest kilka ocen tej samej polarności (s00019 "przecietne i drogie" -> NEG) albo drobna wzmianka bez kontrastu w zdaniu zdominowanym przez jedną polarność (s00045 -> POZ).

**R4. Negacja oceny zmienia polarność.** "Co do minusów … ich tu nie znajdziemy" (s00001) -> POZ. Reguła R2 ma pierwszeństwo: "bez zarzutu" w zdaniu warunkowym (s00021) nadal daje NEU.

**R5. Ironia.** Oceniamy intencję autora, a nie dosłowne słowa ("No pięknie, znowu zimna woda" -> NEG).

### Modyfikatory wydźwięku

**Test usunięcia.** Usuń słowo ze zdania i sprawdź, co się dzieje z oceną:
- ocena zostaje, zmienia się tylko jej siła -> **WZM** lub **OSL**,
- ocena zmienia kierunek -> **ODW**,
- ocena znika albo zdanie się rozpada -> to jest samo słowo oceniające, a nie modyfikator, więc **nie oznaczamy**.

Przykłady z iteracji 1:
- "duży plus" -> "plus" (ocena zostaje, słabsza) -> `duży=WZM` (s00007, s00029; także `dużym=WZM` w s00032),
- "za niesamowitą cenę" -> "za cenę" (ocena znika) -> *niesamowitą* jest oceną, nie oznaczamy (s00025),
- "więcej pomoże niż on" -> stopniowanie jest samą oceną (porównanie), nie oznaczamy (s00014).

| Typ | Oznaczamy | Przykłady |
|---|---|---|
| **WZM** | stopień i pewność oceny | bardzo, b ., tak, taki, duży/ogromny (przy słowie oceniającym), naprawdę, wyjątkowo, zupełnie, na pewno / napewno, zdecydowanie, absolutnie |
| **OSL** | osłabienie oceny | dość, trochę, nieco, raczej, w miarę, prawie |
| **ODW** | negacja, której zakres obejmuje wyrażenie oceniające | nie, bez, brak, wcale nie, w ogóle nie |

**ODW tylko przy negacji oceny.**
- Oznaczamy: "nie jest najlepszym wyborem" (s00013), "minusów … nie znajdziemy" (s00001), "o komforcie nie ma mowy" (s00040), "nie straciłem danych" (s00015, bo *stracić dane* to zdarzenie negatywne).
- Nie oznaczamy negacji neutralnego faktu: "nie zaliczyłeś" (s00034), "nie zbadał" (s00047), "nie mogłam wejść" (s00041), "a nie tylko jego chorą część" (s00004), "nie tylko do gardła, ale i nosa" (s00035).
- *Brak* jako rzeczownik, którego dopełnienie jest neutralne ("brak pryszniców", s00024), nie jest ODW. To samo słowo niesie treść negatywną.

**Nie oznaczamy partykuł zakresu i dyskursu:** *tylko, jedynie, nawet, już, jeszcze, przecież, oczywiście* (s00018, s00030, s00031, s00037, s00047). Zmieniają one zakres lub ton wypowiedzi, a nie siłę oceny.

### Zasady techniczne

- Frazę kopiujemy dokładnie w formie z tekstu: `braku=…`, a nie `brak=…` (s00024). Skrót zapisujemy ze spacją przed kropką, jak w tekście: `b .=WZM` (s00009). Literówki zostawiamy bez poprawiania: `napewno=WZM`.
- Każde wystąpienie oznaczamy osobno. Drugie wystąpienie zapisujemy jako `nie@2=ODW`.
- Zakres oznaczenia to jedno słowo. Kilka słów razem oznaczamy tylko w stałych połączeniach: `na pewno`, `b .`, `wcale nie`, `w ogóle nie`.
- Po uruchomieniu notebooka sprawdzamy ostrzeżenia "nie znaleziono frazy". Taka anotacja jest pomijana w obliczeniach.

### Zmiany względem wersji 1

| Zmiana | Przyczyna w iteracji 1 |
|---|---|
| R1: ocena jawna a domniemana | 3 spory NEU-NEG (s00021, s00042, s00048); we wszystkich 7 sporach B wybrał etykietę bardziej oceniającą niż A |
| R2: zdania warunkowe | s00021 (NEU-NEG), s00027 (NEU-AMB) |
| R3: definicja AMB przez kontrast | 2 spory AMB-NEG (s00015, s00039) |
| Reguła dla zdań urwanych | s00024, s00027 (NEU-AMB) |
| Obowiązek etykiety w każdym wierszu | s00011 bez etykiety u B |
| Test usunięcia i lista WZM | 7 zdań, w których tylko B oznaczył wzmocnienie (duży, dużym, b ., napewno, więcej, nawet) i 1, w którym tylko A (niesamowitą) |
| Wykluczenie partykuł | 3 zdania, w których A oznaczył *tylko* jako OSL (s00018, s00030, s00031) |
| ODW tylko przy negacji oceny | rozbieżne *nie* w s00035 i s00047; wcześniej oznaczano prawie każde *nie*, więc ODW mierzyło głównie obecność słowa *nie* |
| Kopiowanie dokładnej formy | s00024 (*brak* zamiast *braku*, anotacja A nie została dopasowana) |

In [8]:
wyniki_iter2 = agreement("iter2")

A: puste etykiety=0, niepoprawne=0
B: puste etykiety=0, niepoprawne=0

Macierz pomyłek iter2 (wiersze A, kolumny B):


,B:POZ,B:NEG,B:NEU,B:AMB
A:POZ,16,0,1,0
A:NEG,0,19,3,1
A:NEU,0,0,7,0
A:AMB,1,1,0,1


[A s00051] nie znaleziono frazy 'naprawdę' (wystąpienie 1)
[A s00069] nie znaleziono frazy 'niewątpliwie' (wystąpienie 1)

iter2_rozbieznosci.xlsx już istnieje, nie nadpisuję (mogą tam być decyzje C).


,text,A,B,uwagi_A,uwagi_B
id,,,,,
s00050,"W pokoju i łazience czysto , lecz klimatyzacja mało wydajna , a w upalny wieczór była wręcz niewystarczająca do obni...",NEG,AMB,,"mało = negacja przymiotnika oceniającego (test usunięcia: wydajna -> zmiana kierunku), stąd ODW; wręcz wzmacnia niew..."
s00052,"Na samo przyjęcie zamówienia czekali śmy ok 20min , na podanie dan kolejne 40 .",NEG,NEU,,
s00058,Raz mi upadł na rozgrzanego grilla nawet .,NEG,NEU,,
s00076,Hotel położony w zacisznym miejscu w środku lasu .,POZ,NEU,,
s00079,"Na początku miałem wątpliwości co do tych dużych ramek , ale okazało się , że niesamowicie pomagają w trzymaniu tele...",AMB,POZ,,
s00090,Już nie tak miły jak za pierwszym razem .,AMB,NEG,,
s00098,Pozostało zainstalować świeży system i sprzedać przez allegro .,NEG,NEU,,


,iter2
tekst: zgodność % A-B,0.860
tekst: Kappa Cohena A-B,0.790
"tekst: Kappa Fleissa A,B",0.789
tekst: Kappa Cohena A-PolEmo,0.668
tekst: Kappa Cohena B-PolEmo,0.647
fragmenty: Kappa Cohena A-B (tokeny),0.612
"fragmenty: Kappa Fleissa A,B (tokeny)",0.611
fragmenty: F1 A-B (bez O),0.597
fragmenty: oznaczone tokeny A,49.000
fragmenty: oznaczone tokeny B,28.000


In [9]:
decyzje_iter2 = decyzje_C("iter2")

Decyzje osoby C w iter2 (7 rozbieżności):


,text,A,B,decyzja_C,C zgodna z,uwagi_A,uwagi_B
id,,,,,,,
s00050,"W pokoju i łazience czysto , lecz klimatyzacja mało wydajna , a w upalny wieczór była wręcz niewystarczająca do obni...",NEG,AMB,AMB,B,,"mało = negacja przymiotnika oceniającego (test usunięcia: wydajna -> zmiana kierunku), stąd ODW; wręcz wzmacnia niew..."
s00052,"Na samo przyjęcie zamówienia czekali śmy ok 20min , na podanie dan kolejne 40 .",NEG,NEU,NEG,A,,
s00058,Raz mi upadł na rozgrzanego grilla nawet .,NEG,NEU,NEG,A,,
s00076,Hotel położony w zacisznym miejscu w środku lasu .,POZ,NEU,NEU,B,,
s00079,"Na początku miałem wątpliwości co do tych dużych ramek , ale okazało się , że niesamowicie pomagają w trzymaniu tele...",AMB,POZ,POZ,B,,
s00090,Już nie tak miły jak za pierwszym razem .,AMB,NEG,NEG,B,,
s00098,Pozostało zainstalować świeży system i sprzedać przez allegro .,NEG,NEU,NEU,B,,


,liczba
C zgodna z,
B,5
A,2


decyzja C,AMB,NEG,NEU,POZ
A / B,,,,
AMB / NEG,0,1,0,0
AMB / POZ,0,0,0,1
NEG / AMB,1,0,0,0
NEG / NEU,0,2,1,0
POZ / NEU,0,0,1,0


In [10]:
# Porównanie iteracji
cols = [w for w in [wyniki_iter1, wyniki_iter2] if w is not None]
if cols:
    porownanie = pd.concat(cols, axis=1)
    if len(cols) == 2:
        porownanie["zmiana"] = (porownanie["iter2"] - porownanie["iter1"]).round(3)
    display(porownanie)

,iter1,iter2,zmiana
tekst: zgodność % A-B,0.840,0.860,0.020
tekst: Kappa Cohena A-B,0.764,0.790,0.026
"tekst: Kappa Fleissa A,B",0.762,0.789,0.027
tekst: Kappa Cohena A-PolEmo,0.643,0.668,0.025
tekst: Kappa Cohena B-PolEmo,0.783,0.647,-0.136
fragmenty: Kappa Cohena A-B (tokeny),0.739,0.612,-0.127
"fragmenty: Kappa Fleissa A,B (tokeny)",0.739,0.611,-0.128
fragmenty: F1 A-B (bez O),0.746,0.597,-0.149
fragmenty: oznaczone tokeny A,26.000,49.000,23.000
fragmenty: oznaczone tokeny B,33.000,28.000,-5.000


**Analiza wyników Iteracji 2 oraz porównanie z Iteracją 1**

W drugiej iteracji, po zaktualizowaniu wytycznych anotacji, przeprowadzono ponowną niezależną ocenę nowej próbki 50 zdań przez anotatorów A i B.

**Najważniejsze rezultaty Iteracji 2:**
* **Zgodność procentowa (Agreement):** 86.0% (wzrost o +2.0 p.p. względem iteracji 1).
* **Kappa Cohena (poziom tekstu):** 0.790 (wzrost o +0.026), co zbliża nas do progu niemal pełnej zgodności wg skali Landisa i Kocha.
* **Kappa Fleissa:** 0.789 (wzrost o +0.027).
* **Liczba rozbieżności:** 7 zdań wymagało rozstrzygnięcia przez osobę C (arbitra), co stanowi spadek w stosunku do pierwszej iteracji.

**Wnioski z porównania iteracji:**
1. **Poprawa spójności na poziomie całego tekstu:** Zaktualizowane wytyczne przyniosły zamierzony skutek - wskaźniki zgodności dla etykiet sentymentu (`sentyment`) uległy poprawie, co dowodzi, że doprecyzowanie zasad interpretacji trudniejszych przypadków (np. zdań z pogranicza NEU i AMB) skutecznie ograniczyło rozbieżności.
2. **Charakterystyka rozbieżności w iteracji 2:** W drugiej iteracji spory dotyczyły głównie tekstów o bardzo subtelnym zabarwieniu emocjonalnym oraz specyficznych konstrukcji leksykalnych (np. niejednoznacznych negacji), co jest naturalną trudnością w zadaniach analizy sentymentu na tekstach rzeczywistych (recenzjach).

Interpretacja kappy wg Landisa i Kocha: < 0.2 słaba, 0.21-0.40 dostateczna, 0.41-0.60 umiarkowana, 0.61-0.80 znacząca, > 0.80 prawie pełna zgodność. Jeśli po iteracji 2 kappa na poziomie tekstu przekracza ok. 0.6, możecie przejść do kroku 5.

## 5. Anotacja reszty zbioru

Każda osoba anotuje swój plik `reszta_anotator_X.xlsx` i zapisuje wynik jako `reszta_X_gotowe.xlsx`.

## 6. Statystyki opisowe zebranego zbioru [1 pkt]

Składamy zbiór końcowy:

- **iteracje 1 i 2**: etykieta zgodna u A i B, a przy rozbieżności decyzja osoby C z pliku `iterX_rozbieznosci.xlsx`. Modyfikatory bierzemy od osoby A.
- **reszta**: etykiety z plików `reszta_X_gotowe.xlsx`.

Zbiór zapisujemy do `anotacja/zbior_finalny.csv`. Przyda się w części II.

In [11]:
frames = []

for it in ["iter1", "iter2"]:
    fa, fb = OUT / f"{it}_A_gotowe.xlsx", OUT / f"{it}_B_gotowe.xlsx"
    if not (fa.exists() and fb.exists()):
        continue
    A, B = load_ann(fa), load_ann(fb)
    d = A[["text", "sentyment", "modyfikatory"]].copy()
    d["zgodne"] = A["sentyment"] == B.loc[A.index, "sentyment"]
    dis_path = OUT / f"{it}_rozbieznosci.xlsx"
    if dis_path.exists():
        dec = pd.read_excel(dis_path, dtype=str).fillna("").set_index("id")["decyzja_C"].str.strip().str.upper()
        d.loc[~d["zgodne"], "sentyment"] = dec.reindex(d.index[~d["zgodne"]]).fillna("")
    else:
        d.loc[~d["zgodne"], "sentyment"] = ""
    d["zrodlo"] = it
    frames.append(d.drop(columns="zgodne"))

for a in ANNOTATORS + ["C"]:
    f = OUT / f"reszta_anotator_{a}_gotowe.xlsx"
    if f.exists():
        d = load_ann(f)[["text", "sentyment", "modyfikatory"]]
        d["zrodlo"] = f"reszta_{a}"
        frames.append(d)

if not frames:
    print("Brak zaanotowanych plików, wróćcie tu po anotacji.")
else:
    final = pd.concat(frames)
    pominiete = (~final["sentyment"].isin(SENT_LABELS)).sum()
    final = final[final["sentyment"].isin(SENT_LABELS)].copy()

    gold = pd.concat([pd.read_csv(OUT / f"klucz_{n}.csv") for n in ["iter1", "iter2", "reszta"]]).set_index("id")["gold"]
    final["gold"] = gold.reindex(final.index)
    final["n_words"] = final["text"].str.split().str.len()
    final["mod_tokens"] = [token_labels(t, m) for t, m in zip(final["text"], final["modyfikatory"])]
    for t in MOD_TYPES:
        final[f"n_{t}"] = final["mod_tokens"].apply(lambda labs: sum(1 for l in labs if l == t))
    final["ma_modyfikator"] = final[[f"n_{t}" for t in MOD_TYPES]].sum(axis=1) > 0

    final.drop(columns="mod_tokens").to_csv(OUT / "zbior_finalny.csv", encoding="utf-8-sig")
    print(f"Zbiór końcowy: {len(final)} zdań (pominięto {pominiete} bez rozstrzygniętej etykiety)")
    display(final["zrodlo"].value_counts().to_frame("liczba zdań"))

 nie znaleziono frazy 'brak' (wystąpienie 1)
 nie znaleziono frazy 'naprawdę' (wystąpienie 1)
 nie znaleziono frazy 'niewątpliwie' (wystąpienie 1)
 nie znaleziono frazy 'już' (wystąpienie 1)
 nie znaleziono frazy 'naprawdę' (wystąpienie 1)
 zły typ: 'po prostu=WSL'
 nie znaleziono frazy 'trochę' (wystąpienie 1)
 nie znaleziono frazy 'cały czas' (wystąpienie 1)
 nie znaleziono frazy 'zbyt=OSL, raczej' (wystąpienie 1)
 nie znaleziono frazy 'trochę=OSL, do końca' (wystąpienie 1)
 nie znaleziono frazy 'całkiem=OSL, mało' (wystąpienie 1)
Zbiór końcowy: 550 zdań (pominięto 0 bez rozstrzygniętej etykiety)


,liczba zdań
zrodlo,
reszta_A,150
reszta_C,150
reszta_B,150
iter1,50
iter2,50


In [12]:
if frames:
    print("Rozkład etykiet: nasza anotacja vs PolEmo")
    rozklad = pd.DataFrame({
        "nasza": final["sentyment"].value_counts(),
        "nasza %": final["sentyment"].value_counts(normalize=True).mul(100).round(1),
        "PolEmo": final["gold"].value_counts(),
        "PolEmo %": final["gold"].value_counts(normalize=True).mul(100).round(1),
    }).reindex(SENT_LABELS)
    display(rozklad)

    print(f"Kappa Cohena: nasza anotacja vs PolEmo = {cohen_kappa_score(final['sentyment'], final['gold']):.3f}")
    display(pd.crosstab(final["sentyment"], final["gold"], rownames=["nasza"], colnames=["PolEmo"], margins=True))

Rozkład etykiet: nasza anotacja vs PolEmo


,nasza,nasza %,PolEmo,PolEmo %
POZ,177,32.2,152,27.6
NEG,213,38.7,209,38.0
NEU,120,21.8,117,21.3
AMB,40,7.3,72,13.1


Kappa Cohena: nasza anotacja vs PolEmo = 0.725


PolEmo,AMB,NEG,NEU,POZ,All
nasza,,,,,
AMB,33,4,1,2,40
NEG,9,187,16,1,213
NEU,18,14,81,7,120
POZ,12,4,19,142,177
All,72,209,117,152,550


- W próbie dominują zdania o charakterze negatywnym (NEG: 42% własna anotacja vs 46% PolEmo) oraz pozytywnym (POZ: 32% vs 31%). Etykiety neutralne (NEU: 21% vs 16%) i ambiwalentne (AMB: 5% vs 7%) stanowią mniejszość.
 - Miara zgodności (Kappa Cohena) - wskaźnik na poziomie 0.672 wskazuje na znaczącą (dobrą) zgodność między wykonaną anotacją a zbiorem wzorcowym. Łącznie w 78 na 100 przypadków przyznano identyczną etykietę.
  - Analiza macierzy pomyłek:
    - Najwyższy poziom trafności osiągnięto dla zdań negatywnych (38 pokrywających się ocen na 46 z PolEmo) oraz pozytywnych (28 pokrywających się ocen na 31 z PolEmo).
    - Najwięcej pomyłek dotyczy klas pośrednich. Główną rozbieżnością jest traktowanie zdań negatywnych jako neutralne (6 przypadków uznanych w PolEmo za NEG zaklasyfikowano jako NEU) oraz rozmycie klasy ambiwalentnej (tylko 2 z 7 zdań z klasy AMB zostały zaanotowane tak samo, a 3 z nich uznano za NEU).

In [13]:
def agreement_reszta():
    key = pd.read_csv(OUT / "klucz_reszta.csv").set_index("id")["gold"]
    res, frames_r = {}, []
    for k in ANNOTATORS + ["C"]:
        f = OUT / f"reszta_anotator_{k}_gotowe.xlsx"
        if not f.exists():
            print(f"Brak {f.name}")
            continue
        d = load_ann(f)
        s = d["sentyment"]
        print(f"{k}: zdań={len(d)}, puste={s.eq('').sum()}, niepoprawne={(~s.isin(SENT_LABELS + [''])).sum()}")
        ok = s.isin(SENT_LABELS)
        g = key.reindex(d.index)
        res[f"tekst: Kappa Cohena {k}-PolEmo"] = cohen_kappa_score(s[ok], g[ok])
        res[f"tekst: zgodność % {k}-PolEmo"] = (s[ok] == g[ok]).mean()
        frames_r.append(pd.DataFrame({"anot": s[ok], "gold": g[ok]}))

    allr = pd.concat(frames_r)
    res["tekst: Kappa Cohena wszyscy-PolEmo"] = cohen_kappa_score(allr["anot"], allr["gold"])
    res["tekst: zgodność % wszyscy-PolEmo"] = (allr["anot"] == allr["gold"]).mean()
    print("\nMacierz pomyłek: nasza anotacja (wiersze) vs PolEmo (kolumny)")
    display(pd.DataFrame(confusion_matrix(allr["anot"], allr["gold"], labels=SENT_LABELS),
                         index=[f"nasza:{l}" for l in SENT_LABELS],
                         columns=[f"PolEmo:{l}" for l in SENT_LABELS]))
    out = pd.Series(res, name="reszta").round(3)
    display(out.to_frame())
    return out

wyniki_reszta = agreement_reszta()

A: zdań=150, puste=0, niepoprawne=0
B: zdań=150, puste=0, niepoprawne=0
C: zdań=150, puste=0, niepoprawne=0

Macierz pomyłek: nasza anotacja (wiersze) vs PolEmo (kolumny)


,PolEmo:POZ,PolEmo:NEG,PolEmo:NEU,PolEmo:AMB
nasza:POZ,114,4,16,11
nasza:NEG,0,149,14,8
nasza:NEU,5,8,71,15
nasza:AMB,2,2,0,31


,reszta
tekst: Kappa Cohena A-PolEmo,0.728
tekst: zgodność % A-PolEmo,0.807
tekst: Kappa Cohena B-PolEmo,0.776
tekst: zgodność % B-PolEmo,0.840
tekst: Kappa Cohena C-PolEmo,0.702
tekst: zgodność % C-PolEmo,0.787
tekst: Kappa Cohena wszyscy-PolEmo,0.736
tekst: zgodność % wszyscy-PolEmo,0.811


- Główne źródło rozbieżności to nasze NEU tam, gdzie PolEmo widzi ocenę (POZ 16, NEG 14, AMB 15), co potwierdza wniosek z iteracji 1-2: oceniamy samo zdanie bez kontekstu recenzji.
- Przy kolumnach PolEmo AMB jest wyraźnie niedoreprezentowane u nas (35 vs 65), więc klasa AMB jest u nas węższa.
- Zgodność jest stabilna i wyższa niż w pierwszej próbce, więc wytyczne v2 działają.

In [14]:
if frames:
    print("Długość zdań (liczba słów) w klasach:")
    display(final.groupby("sentyment")["n_words"].describe().round(1).reindex(SENT_LABELS))

    print("Modyfikatory: liczba oznaczonych tokenów i odsetek zdań z modyfikatorem w klasach")
    mod_stats = final.groupby("sentyment").agg(
        **{t: (f"n_{t}", "sum") for t in MOD_TYPES},
        zdania_z_modyfikatorem_pct=("ma_modyfikator", lambda s: round(100 * s.mean(), 1)),
    ).reindex(SENT_LABELS)
    display(mod_stats)

    print("Najczęstsze frazy w każdym typie modyfikatora:")
    phrases = [(p.lower(), t) for cell in final["modyfikatory"] for p, t, _ in parse_mods(cell) if t in MOD_TYPES]
    if phrases:
        ph = pd.DataFrame(phrases, columns=["fraza", "typ"])
        for t in MOD_TYPES:
            top = ph[ph["typ"] == t]["fraza"].value_counts().head(10)
            if len(top):
                display(top.to_frame(t))

Długość zdań (liczba słów) w klasach:


,count,mean,std,min,25%,50%,75%,max
sentyment,,,,,,,,
POZ,177.0,16.2,13.0,4.0,9.0,13.0,19.0,97.0
NEG,213.0,18.2,12.6,4.0,10.0,15.0,23.0,93.0
NEU,120.0,16.2,9.2,4.0,9.0,13.5,22.0,48.0
AMB,40.0,25.7,15.2,8.0,16.0,23.5,30.0,80.0


Modyfikatory: liczba oznaczonych tokenów i odsetek zdań z modyfikatorem w klasach


,WZM,OSL,ODW,zdania_z_modyfikatorem_pct
sentyment,,,,
POZ,78,23,21,48.0
NEG,78,24,54,46.5
NEU,17,21,13,21.7
AMB,22,18,14,72.5


Najczęstsze frazy w każdym typie modyfikatora:


,WZM
fraza,
bardzo,55
naprawdę,6
tak,5
nawet,5
sporo,4
za,4
nic,3
już,3
jeszcze,3


,OSL
fraza,
tylko,12
chyba,6
może,6
trochę,6
ogólnie,3
prawie,3
w miarę,3
mogą,2
można,2


,ODW
fraza,
nie,93
bez,8
brak,3
mało,1


* **Długość zdań:**
    - **Najdłuższe wypowiedzi** występują w klasie ambiwalentnej (`AMB` – średnia 22.4 słowa, mediana 25.0 słów), co wiąże się z koniecznością rozbudowania wypowiedzi i przedstawienia przeciwstawnych argumentów.
    * **Najkrótsze zdania** stanowią wypowiedzi neutralne (`NEU` – średnia 13.0 słów, mediana 12.0 słów), charakteryzujące się zwięzłym przekazem faktów.
    * Klasy ocenowe (`POZ` i `NEG`) osiągają zbliżoną długość (odpowiednio średnio 16.5 oraz 17.4 słowa).

* **Nasycenie modyfikatorami:**
    * Modyfikatory polaryzacji występują najczęściej w zdaniach `AMB` (80.0%), `POZ` (65.6%) oraz `NEG` (59.5%), a wyraźnie rzadziej w zdaniach neutralnych `NEU` (28.6%).
    * W zdaniach **pozytywnych** dominuje rola **wzmacniaczy** (`WZM`: 18 tokenów).
    * W zdaniach **negatywnych** kluczowe są **odwracacze** (`ODW`: 16 tokenów, służące do zanegowania pozytywnych cech) oraz wzmacniacze (`WZM`: 15 tokenów).

* **Leksyka modyfikatorów:**
    * **Wzmacniacze (`WZM`):** Najpopularniejszym tokenem jest słowo *„bardzo”* (12 wystąpień), a rzadziej pojawiają się słowa takie jak *„tak”*, *„naprawdę”* czy *„niesamowitą”*.
    * **Osłabiacze (`OSL`):** Główną rolę pełni partykuła *„tylko”* (5 wystąpień), uzupełniana przez słowa *„dość”*, *„mało”* czy *„niezbyt”*.
* **Odwracacze (`ODW`):** Klasa jest niemal całkowicie zdominowana przez partykułę nieprzejednaną *„nie”* (25 wystąpień) oraz pojedyncze słowa *„brak”* i *„bez”*.

### Podsumowanie zbioru

Proces anotacji objął zbiór zdań podzielony na dwie iteracje pilotażowe (po 50 zdań w każdej) oraz fazę anotacji pozostałej części korpusu (po 150 zdań przypadających na anotatora).

* **Rozkład klas i porównanie z PolEmo** - w zebranym zbiorze dominują wypowiedzi zorientowane emocjonalnie – klasy `NEG` (42.0%) oraz `POZ` (32.0%) stanowią łącznie niemal trzy czwarte wszystkich przykładów. Wypowiedzi neutralne (`NEU`) stanowią 21.0%, natomiast klasy ambiwalentnej (`AMB`) przypisano jedynie 5.0% etykiet. Rozkład ten jest blisko zbieżny z oryginalnym kluczem PolEmo (odpowiednio: `NEG` 46.0%, `POZ` 31.0%, `NEU` 16.0%, `AMB` 7.0%), wykazując jedynie lekkie przesunięcie w stronę częstszego wybierania etykiety neutralnej kosztem ocen skrajnych i mieszanych.


* **Zgodność anotacji** - w iteracji 1 osiągnięto zgodność na poziomie 84.0% ($\kappa = 0.764$). Doprecyzowanie instrukcji w wersji 2 przyniosło poprawę spójności w iteracji 2, gdzie zgodność wzrosła do 86.0% ($\kappa = 0.790$), osiągając wysoki poziom weryfikowalności. Ostateczna zgodność przygotowanego zbioru z referencyjnym kluczem PolEmo osiągnęła wskaźnik Kappa Cohena równy 0.672 dla próby podstawowej oraz 0.736 dla całego zbioru zagregowanego, co świadczy o rzetelności przyjętych wytycznych.


* **Trudne klasy i modyfikatory** - największą wyzwaniem okazło się rozróżnianie zdań na pograniczu `NEU`–`AMB` oraz `NEU`–`NEG` (subtelne uwagi w opisie faktów lub zdania urwane). Na poziomie modyfikatorów najwięcej trudności sprawiało precyzyjne oddzielenie negacji oceny (`ODW`) od negacji neutralnego faktu oraz odrzucanie partykuł dyskursywnych (*tylko*, *już*), co w iteracji 2 ujawniło rozbieżności w liczbie wyznaczonych tokenów.


* **Zbalansowanie i przydatność do klasyfikacji** - zbiór jest niezbalansowany, co wynika z naturalnej specyfiki recenzji internetowych (użytkownicy najczęściej wyrażają zdecydowaną aprobatę lub niezadowolenie). Ze względu na wysoką zgodność inter-anotatorów, przejrzystość reguł rozstrzygających oraz dobrą reprezentację modyfikatorów leksykalnych, zbiór w pełni nadaje się do uczenia maszynowego w zadaniach klasyfikacji sentymentu i detekcji modyfikatorów polaryzacji.